In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sympy.integrals.manualintegrate import TrigRule
from webencodings import labels

## 6.2 Dataset Preparation

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download "
              "and extraction."
        )
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [ ]:
import pandas as pd
df = pd.read_csv(
    data_file_path, sep="\t", header=None, names=["Label", "Text"]
)
df

In [ ]:
df.head()

In [ ]:
print(df["Label"].value_counts())

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam, random_state=123
    )

    balanced_df = pd.concat([
        ham_subset, df[df["Label"] == "spam"]
    ])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

In [ ]:
def create_unbalanced_dataset(df):
    ham_subset = df[df["Label"] == "ham"]
    spam_subset = df[df["Label"] == "spam"]

    unbalanced_df = pd.concat([ham_subset, spam_subset])

    return unbalanced_df

unbalanced_df = create_unbalanced_dataset(df)
print(unbalanced_df["Label"].value_counts())

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
unbalanced_df["Label"] = unbalanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
def random_split(df, train_frac, validation_frac):

    df = df.sample(
        frac=1, random_state=123
    ).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)


    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_ub_df, validation_ub_df, test_ub_df = random_split(unbalanced_df, 0.7, 0.1)


## 6.3 Creating Datasets

## 6.3 Create Data Loaders

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None,
                 pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length

            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]


        self.encoded_texts = [
            encoded_text + [pad_token_id] *
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]


    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

In [ ]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

train_ub_dataset = SpamDataset(
    csv_file="train_ub.csv",
    max_length=None,
    tokenizer=tokenizer
)

In [ ]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
val_ub_dataset = SpamDataset(
    csv_file="validation_ub.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_ub_dataset = SpamDataset(
    csv_file="test_ub.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
train_ub_loader = DataLoader(
    dataset=train_ub_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_ub_loader = DataLoader(
    dataset=val_ub_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_ub_loader = DataLoader(
    dataset=test_ub_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

## 6.4 Initializing model with pretrained weights.

In [39]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [40]:
BASE_CONFIG

{'vocab_size': 50257,
 'context_length': 1024,
 'drop_rate': 0.0,
 'qkv_bias': True,
 'emb_dim': 768,
 'n_layers': 12,
 'n_heads': 12}

In [42]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size, models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 7.25kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:02<00:00, 381kiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 12.8kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [02:31<00:00, 3.28MiB/s]   
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 358kiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:02<00:00, 224kiB/s]  
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 253kiB/s]  


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [43]:
from previous_chapters import generate_text_simple
from previous_chapters import text_to_token_ids, token_ids_to_text

text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [44]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


In [ ]:
## 6.5 Adding a classification head

In [45]:
model

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [47]:
for param in model.parameters():
    param.requires_grad = False

In [49]:
model.tok_emb

Embedding(50257, 768)

In [53]:
model.trf_blocks[0].att

MultiHeadAttention(
  (W_query): Linear(in_features=768, out_features=768, bias=True)
  (W_key): Linear(in_features=768, out_features=768, bias=True)
  (W_value): Linear(in_features=768, out_features=768, bias=True)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)

In [54]:
torch.manual_seed(123)
num_classes = 2
model.out_head = nn.Linear(in_features = BASE_CONFIG["emb_dim"], out_features = num_classes)


In [55]:
model.out_head

Linear(in_features=768, out_features=2, bias=True)

In [56]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [57]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)

Inputs: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])


In [58]:
with torch.no_grad():
    outputs = model(inputs)
print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape)

Outputs:
 tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])
Outputs dimensions: torch.Size([1, 4, 2])


In [59]:
print("Last output token:", outputs[:, -1, :])

Last output token: tensor([[-3.5983,  3.9902]])


In [ ]:
## 6.6Calculating the classification loss and accuracy

In [63]:
probas = F.softmax(outputs[:, -1, :], dim=-1)
labels = torch.argmax(probas, dim=-1)
labels

tensor([1])

In [64]:
print("Class label:", labels[0].item())

Class label: 1


In [65]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)


            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (
                (predicted_labels == target_batch).sum().item()
            )

        else:
            break
    return correct_predictions / num_examples

In [68]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [69]:
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(
    train_loader, model, device, num_batches=10
)
val_accuracy = calc_accuracy_loader(
    val_loader, model, device, num_batches=10
)
test_accuracy = calc_accuracy_loader(
    test_loader, model, device, num_batches=10
)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 46.25%
Validation accuracy: 45.00%
Test accuracy: 48.75%


In [70]:
torch.manual_seed(123)
train_ub_accuracy = calc_accuracy_loader(
    train_ub_loader, model, device, num_batches=10
)
val_ub_accuracy = calc_accuracy_loader(
    val_ub_loader, model, device, num_batches=10
)
test_ub_accuracy = calc_accuracy_loader(
    test_ub_loader, model, device, num_batches=10
)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 46.25%
Validation accuracy: 45.00%
Test accuracy: 48.75%


## 6.4.A Focal Loss Implementation

Link to the focal loss paper: https://github.com/itakurah/focal-loss-pytorch

In [ ]:
y = torch.randn((8,3))
y

In [ ]:
rows = [1,2]
cols = [0,1]

y[rows,cols]

In [ ]:
class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss for per-token classification.

        FL(p_t) = - alpha_t * (1 - p_t)**gamma * log(p_t)

    Inputs are token-level logits of shape (batch_size, seq_len, num_classes)
    and targets are class indices of shape (batch_size, seq_len). Both are
    flattened to (batch_size * seq_len, ...) before the focal-loss math.

    Setting alpha proportional to *inverse class frequency* up-weights minority
    classes so that rare-but-hard tokens drive the gradient.
    """

    def __init__(self, gamma=2.0, alpha=None):
        """
        :param gamma: focusing parameter; down-weights easy (high p_t) examples.
        :param alpha: class balancing factor. One of:
            - None: no class weighting (alpha_t = 1 for every token).
            - float: binary case; weight for class 1, (1 - alpha) for class 0.
            - 1-D tensor of shape (num_classes,): per-class weights.
              Use FocalLoss.alpha_from_counts(counts) for inverse-frequency weights.
        """
        super().__init__()
        self.gamma = gamma
        if torch.is_tensor(alpha):
            # Register as buffer so it moves with .to(device) and saves with the module.
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha = alpha  # None or float scalar

    @staticmethod
    def alpha_from_counts(counts):
        """
        Inverse-frequency class weights (sklearn "balanced" convention):

            alpha[c] = N_total / (num_classes * count[c])

        Properties:
          - For a perfectly balanced dataset every alpha[c] == 1.
          - Minority classes get alpha > 1, majority classes get alpha < 1.
          - sum(count[c] * alpha[c]) == N_total, so each class contributes
            an equal *expected* weight to the loss.
        """
        counts = torch.as_tensor(counts, dtype=torch.float32)
        num_classes = counts.numel()
        n_total = counts.sum()
        return n_total / (num_classes * counts)

    def forward(self, inputs, targets):
        """
        :param inputs:  logits of shape (batch_size, seq_len, num_classes)
        :param targets: class indices of shape (batch_size, seq_len)
        """
        num_classes = inputs.shape[-1]
        # Collapse the (batch, seq) axes so every token becomes an independent sample.
        inputs_flat  = inputs.reshape(-1, num_classes)    # (B*S, C)
        targets_flat = targets.reshape(-1)                # (B*S,)

        # Per-token cross-entropy = -log p_t for the true class.
        ce = F.cross_entropy(inputs_flat, targets_flat, reduction="none")
        # Recover p_t = exp(-ce). The original code did exp(ce), which gives 1/p_t.
        p_t = torch.exp(-ce)

        # Per-token alpha_t
        if self.alpha is None:
            alpha_t = 1.0
        elif torch.is_tensor(self.alpha):
            alpha_t = self.alpha[targets_flat]
        else:
            # Binary scalar: alpha for class 1, (1 - alpha) for class 0.
            alpha_t = torch.where(
                targets_flat == 1,
                torch.full_like(ce, self.alpha),
                torch.full_like(ce, 1.0 - self.alpha),
            )

        # Focal term * NLL. ce already equals -log(p_t), so no leading minus.
        loss = alpha_t * (1.0 - p_t) ** self.gamma * ce
        return loss.mean()

### Using inverse-frequency alpha on the SMS spam dataset

The full corpus has **4825 ham** vs **747 spam** messages. Plugging those into
`FocalLoss.alpha_from_counts` gives:

- `alpha[ham]  = 5572 / (2 * 4825) ≈ 0.577`  (majority → down-weighted)
- `alpha[spam] = 5572 / (2 *  747) ≈ 3.730`  (minority → up-weighted)

So a misclassified spam token contributes ~6.5× more gradient than a misclassified
ham token, which is exactly the imbalance ratio. For the balanced training split
(747 ham, 747 spam) both weights are 1.0 — i.e. focal loss with this alpha
gracefully degenerates to the un-weighted version when the data is already balanced.

In [ ]:
# Class counts straight from the unbalanced SMS spam corpus
# (index 0 = ham, index 1 = spam, matching the eventual label encoding)
class_counts = [
    int((df["Label"] == "ham").sum()),
    int((df["Label"] == "spam").sum()),
]
print("class counts:", class_counts)

alpha = FocalLoss.alpha_from_counts(class_counts)
print("inverse-frequency alpha:", alpha)

criterion = FocalLoss(gamma=2.0, alpha=alpha)
print(criterion)

# Smoke test on dummy per-token logits/targets: (B, S, C) and (B, S)
torch.manual_seed(0)
batch_size, seq_len, num_classes = 4, 16, 2
logits  = torch.randn(batch_size, seq_len, num_classes, requires_grad=True)
targets = torch.randint(0, num_classes, (batch_size, seq_len))
loss = criterion(logits, targets)
loss.backward()
print(f"focal loss = {loss.item():.4f}, grad ok: {logits.grad is not None}")